In [ ]:
import os
import pandas as pd
from huggingface_hub import InferenceClient
from transformers import AutoTokenizer

# Initialize client
client = InferenceClient(
    provider="hf-inference",
    api_key=""
)

# Load tokenizer for proper truncation (same model base)
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

In [10]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_114.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)

# Filter data for February and March
# df = df[df['datetime'].between('2016-01-01', '2016-12-31')]
df = df[df['Chamber'] == 'House']
TEST_N = 10
df = df.head(TEST_N).copy() 
republican_emails = df[df['Party'] == 'Republican']
democrat_emails = df[df['Party'] == 'Democrat']

/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_12906/1595116338.py:1: DtypeWarning: Columns (2,4,10,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,25

In [11]:
import time
from tqdm import tqdm
import re

# Set dry-run to True to skip API calls and inspect chunking
DRY_RUN = False

def split_into_token_chunks(text, max_tokens=510, overlap=50):
    """Split `text` into token-id chunks using the tokenizer.
    Returns a list of decoded text chunks. Uses `overlap` tokens of overlap to
    preserve a small amount of context between chunks. If tokenization fails or
    produced very short output, returns an empty list.
    """
    # Normalize input
    text = str(text) if pd.notna(text) else ""
    if not text or not text.strip():
        return []
    text = text.replace('\x00', '').replace('\ufffd', '').strip()

    try:
        token_ids = tokenizer.encode(text, add_special_tokens=False)
    except Exception as e:
        # Tokenization failed; fallback to splitting on paragraphs/sentences
        return [c.strip() for c in re.split('(?<=\n\n)|(?<=\. )', text) if len(c.strip()) >= 10][:10]

    if not token_ids:
        return []

    chunks = []
    step = max_tokens - overlap if max_tokens > overlap else max_tokens
    for start in range(0, len(token_ids), step):
        chunk_ids = token_ids[start:start + max_tokens]
        if not chunk_ids:
            continue
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunk_text = re.sub(r'\s+', ' ', chunk_text).strip()
        if chunk_text and len(chunk_text) >= 10:
            chunks.append(chunk_text)
    return chunks

# Initialize empty list to store results (one entry per chunk)
results = []

# Process each email and split into token chunks when needed
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing emails"):
    try:
        body = row.get('Body', '')
        chunks = split_into_token_chunks(body, max_tokens=510, overlap=50)

        # If chunking produced nothing useful, record a single row without scores
        if not chunks:
            row_data = {
                'date': row['datetime'],
                'first_name': row['First Name'],
                'last_name': row['Last Name'],
                'subject': row['Subject'],
                'id': row['ID'],
                'chunk_index': None,
                'chunk_count': 0,
                'chunk_text': None,
                'chunk_token_count': 0,
            }
            results.append(row_data)
            continue

        # Classify each chunk and add as its own row (so long emails produce multiple rows)
        for cidx, chunk_text in enumerate(chunks, start=1):
            # compute token count for this chunk (useful for weighted averaging later)
            try:
                chunk_token_count = len(tokenizer.encode(chunk_text, add_special_tokens=False))
            except Exception:
                chunk_token_count = None

            # If DRY_RUN is True, skip the HF API call and just record chunk info
            if DRY_RUN:
                emotion_result = []
            else:
                try:
                    emotion_result = client.text_classification(
                        chunk_text,
                        model="j-hartmann/emotion-english-distilroberta-base",
                    )
                except Exception as e:
                    print(f"Error classifying chunk {cidx} for row {idx}: {e}")
                    emotion_result = []

            row_data = {
                'date': row['datetime'],
                'first_name': row['First Name'],
                'last_name': row['Last Name'],
                'subject': row['Subject'],
                'id': row['ID'],
                'chunk_index': cidx,
                'chunk_count': len(chunks),
                'chunk_text': chunk_text if DRY_RUN else None,
                'chunk_token_count': chunk_token_count,
            }

            for emotion in emotion_result:
                emotion_key = f"{emotion['label'].lower()}_score"
                row_data[emotion_key] = emotion['score']

            results.append(row_data)
            # Small delay to avoid rate limiting per chunk (no-op in DRY_RUN)
            time.sleep(0.1)

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        row_data = {
            'date': row['datetime'],
            'first_name': row['First Name'],
            'last_name': row['Last Name'],
            'subject': row['Subject'],
            'id': row['ID'],
            'chunk_index': None,
            'chunk_count': 0,
            'chunk_text': None,
            'chunk_token_count': 0,
        }
        results.append(row_data)

# Create dataframe from results
emotion_results_df = pd.DataFrame(results)

# If DRY_RUN, show a quick sample of chunking output so you can inspect behavior
if DRY_RUN:
    display_cols = ['id', 'chunk_index', 'chunk_count', 'chunk_token_count', 'chunk_text']
    print("DRY_RUN: sample chunk rows")
    try:
        print(emotion_results_df[display_cols].head(20).to_string(index=False))
    except Exception:
        # If chunk_text is large, avoid printing too much
        print(emotion_results_df[[c for c in display_cols if c in emotion_results_df.columns]].head(20))

# Compute per-email averaged emotion scores (mean across chunks)
score_cols = [c for c in emotion_results_df.columns if c.endswith('_score')]
if score_cols:
    averaged_df = (
        emotion_results_df
        .groupby(['id', 'date', 'first_name', 'last_name', 'subject'])[score_cols]
        .mean()
        .reset_index()
    )
    # Add chunk counts per email
    chunk_counts = (
        emotion_results_df.groupby('id')['chunk_count'].max().reset_index(name='num_chunks')
    )
    averaged_df = averaged_df.merge(chunk_counts, on='id', how='left')
else:
    averaged_df = pd.DataFrame()

emotion_results_df.head()


Processing emails:   0%|          | 0/10 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1009 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1009 > 512). Running this sequence through the model will result in indexing errors
Processing emails: 100%|██████████| 10/10 [00:05<00:00,  1.93it/s]



,date,first_name,last_name,subject,id,chunk_index,chunk_count,chunk_text,chunk_token_count,neutral_score,anger_score,joy_score,disgust_score,sadness_score,fear_score,surprise_score
0,2015-01-03 10:13:20,Jason,Smith,Congressman Jason Smith Capitol Report: The Ye...,101001,1,3,None,510,0.816557,0.077309,0.065098,0.016893,0.013856,NaN,NaN
1,2015-01-03 10:13:20,Jason,Smith,Congressman Jason Smith Capitol Report: The Ye...,101001,2,3,None,509,0.055210,0.412334,0.007870,0.007087,NaN,0.507863,NaN
2,2015-01-03 10:13:20,Jason,Smith,Congressman Jason Smith Capitol Report: The Ye...,101001,3,3,None,88,0.936415,0.008600,0.004651,NaN,0.009899,NaN,0.035903
3,2015-01-03 10:13:20,Henry,Cuellar,Congressional Report,101000,1,4,None,510,0.295221,0.005299,0.611964,NaN,0.010520,NaN,0.072892
4,2015-01-03 10:13:20,Henry,Cuellar,Congressional Report,101000,2,4,None,510,0.762456,NaN,0.079897,NaN,0.052898,0.035221,0.059500


In [12]:
emotion_results_df.info()
emotion_results_df.to_csv('../data/emotion_scores/emotion_scores_dcinbox_114.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               20 non-null     datetime64[ns]
 1   first_name         20 non-null     object        
 2   last_name          20 non-null     object        
 3   subject            20 non-null     object        
 4   id                 20 non-null     int64         
 5   chunk_index        20 non-null     int64         
 6   chunk_count        20 non-null     int64         
 7   chunk_text         0 non-null      object        
 8   chunk_token_count  20 non-null     int64         
 9   neutral_score      19 non-null     float64       
 10  anger_score        15 non-null     float64       
 11  joy_score          13 non-null     float64       
 12  disgust_score      5 non-null      float64       
 13  sadness_score      17 non-null     float64       
 14  fear_score  